# Guide 10 — The Control Panel (buttons & screen)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

This builds the **control panel** you actually click during a match: the Connect
button, Start/Stop, the "Wait Mode / Play Mode" switch, a box to ask for a hint, and
a live scoreboard. Once you're in Play Mode, the program plays by itself — there's no
"flip now" button.

This is real code and it's the longest guide, because a control panel has a lot of
buttons. You don't need to memorize it; skim it and notice the pattern: **make a
button → write what happens when it's clicked → show it on screen.**


### How this guide fits in

**Depends on:** Guides 1 and 7, and shows results from all the others. **Used by:** you (it's the buttons you click).

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### The whole control panel

Everything below builds the buttons and the live status display, then connects each
button to the action it should do. Read the section headers in the code (Connection,
Match Control, Status, Hint, Log) to find your way around.

One neat detail worth spotting: the status display only ever *updates* text — it
never clears the screen — so the panel never disappears while the game updates it in
the background.


In [ ]:
server_text = widgets.Text(value=SERVER, description='Server:')
key_text = widgets.Text(value=BROKER_KEY, description='Key:')
referee_text = widgets.Text(value=REFEREE_ID, description='Referee ID:')
master_text = widgets.Text(value=MASTER_ID, description='Master ID:')
team_text = widgets.Text(value=TEAM_NAME, description='Team:')
team_secret_text = widgets.Text(value=TEAM_SECRET, description='Team secret:',
                                 placeholder='leave blank to skip self-reporting your MAC')
board_id_text = widgets.Text(value=BOARD_ID_OVERRIDE, description='Board ID override:',
                              placeholder='leave blank to use pynqp2p.get_id()')

test_connectivity_button = widgets.Button(description='Test Connectivity', button_style='info')
connect_button = widgets.Button(description='Connect', button_style='primary')
disconnect_button = widgets.Button(description='Disconnect', button_style='danger', disabled=True)
start_button = widgets.Button(description='Start', button_style='success', disabled=True)
pause_button = widgets.Button(description='Pause', disabled=True)
resume_button = widgets.Button(description='Resume', disabled=True)
stop_button = widgets.Button(description='Stop', button_style='danger', disabled=True)

control_mode_toggle = widgets.ToggleButtons(
    options=[('Manual', 'manual'), ('Auto', 'auto')],
    value='manual', description='Control:',
)
play_mode_toggle = widgets.ToggleButtons(
    options=[('Wait Mode', False), ('Play Mode', True)],
    value=False, description='Mode:', disabled=True,
)

hint_object_text = widgets.Text(value='', placeholder='e.g. dog', description='Hint object:')
hint_button = widgets.Button(description='Queue Hint', button_style='warning', disabled=True)

solve_riddle_button = widgets.Button(description='Solve Riddle (LLM)', button_style='info')

camera_debug_position_text = widgets.Text(value='A1', description='Grid cell:', layout=widgets.Layout(width='180px'))
camera_debug_button = widgets.Button(description='Debug Oriented Cell', button_style='info')
camera_debug_output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

stage_html = widgets.HTML(value='')
status_html = widgets.HTML(value='<i>Not connected.</i>')
board_html = widgets.HTML(value='')
show_raw_log_checkbox = widgets.Checkbox(value=False, description='Show raw wire messages', indent=False)
# HTML value updates are safe from the background poll thread. Output +
# clear_output() here could clear the parent cell and make the dashboard vanish.
log_html = widgets.HTML(
    value='<i>Enable raw wire messages to view the communication log.</i>',
    layout=widgets.Layout(border='1px solid #ddd', height='160px', overflow_y='auto'),
)
action_output = widgets.Output()
extended_tools_panel = widgets.VBox([])  # Section 12 inserts MNIST here without a second display()

match = None  # the active MatchClient, set on Connect

STAGE_COLOR_DONE = '#2ea043'
STAGE_COLOR_ACTIVE = '#d29922'
STAGE_COLOR_PENDING = '#30363d'


def render_stage_tracker(stage):
    """A fixed, always-visible breadcrumb of named stages -- replaces
    free-form status prose ("Your turn (Play Mode). Flips this turn:
    1/2...") with something a student or spectator can read at a glance
    without parsing a sentence. Pregame stages run once; PLAY_*/WAIT_*
    alternate every turn depending on whose turn it currently is."""
    if stage == Stage.GAME_OVER:
        return (
            '<span style="background:#8250df;color:#fff;padding:4px 10px;'
            'border-radius:4px;font-size:12px;font-weight:bold">Game Over</span>'
        )

    stage_keys = [key for key, _ in PLAY_STAGES]
    if stage in stage_keys:
        stages = PLAY_STAGES
        label_prefix = 'Play: '
    else:
        stage_keys = [key for key, _ in WAIT_STAGES]
        if stage in stage_keys:
            stages = WAIT_STAGES
            label_prefix = 'Wait: '
        else:
            stages = PREGAME_STAGES
            label_prefix = ''

    keys = [key for key, _ in stages]
    current_idx = keys.index(stage) if stage in keys else -1
    chips = []
    for idx, (_, label) in enumerate(stages):
        if idx < current_idx:
            color = STAGE_COLOR_DONE
        elif idx == current_idx:
            color = STAGE_COLOR_ACTIVE
        else:
            color = STAGE_COLOR_PENDING
        chips.append(
            f'<span style="background:{color};color:#fff;padding:4px 10px;'
            f'border-radius:4px;margin-right:4px;font-size:12px">{label}</span>'
        )
    prefix = f'<b>{label_prefix}</b>' if label_prefix else ''
    return prefix + ''.join(chips)


def render_status():
    if match is None:
        stage_html.value = ''
        status_html.value = '<i>Not connected.</i>'
        board_html.value = ''
        return

    stage_html.value = render_stage_tracker(match.stage)

    flip_banner = ''
    if match.last_revealed_pos:
        flip_banner = (
            '<div style="background:#ffcc00;color:#000;padding:10px;font-size:20px;'
            'font-weight:bold;text-align:center;margin-bottom:8px">'
            f'FLIP THE PHYSICAL CARD AT {match.last_revealed_pos} NOW</div>'
        )

    scores_text = ', '.join(f'{team}: {score}' for team, score in match.scores.items()) or 'no score yet'
    pairs_text = (
        f"{match.total_pairs - match.pairs_remaining}/{match.total_pairs} pairs"
        if match.pairs_remaining is not None else f"0/{match.total_pairs} pairs"
    )
    if match.current_turn:
        flip_num = match.current_turn.get('flip_num')
        flip_num_text = f', server flip {flip_num}' if flip_num is not None else ''
        turn_text = (
            f"<br><b>Current turn:</b> #{match.current_turn['turn_number']} — "
            f"{match.current_turn['active_team']}{flip_num_text}"
        )
    else:
        turn_text = '<br><b>Current turn:</b> waiting for the server'
    selected_text = (
        f"<br><b>Cards selected:</b> {', '.join(match.selected_positions)}"
        if match.selected_positions else '<br><b>Cards selected:</b> none yet'
    )
    memory_text = (
        f'<br><b>Memory:</b> {len(match.board_memory)} revealed cells, '
        f'{len(match.reveal_history)} reveal records, '
        f'{len(match.confirmed_matches)} confirmed matches'
    )
    hint_text = f'<br>Last hint: {match.last_hint}' if match.last_hint else ''
    queued_text = f'<br>Hint request pending at referee: {match.pending_hint_object}' if match.pending_hint_object else ''
    hint_images = ''
    if match.last_hint_row_png_base64 and match.last_hint_col_png_base64:
        hint_images = (
            '<br>Row: <img style="vertical-align:middle;background:#fff" '
            f'src="data:image/png;base64,{match.last_hint_row_png_base64}">'
            '&nbsp;&nbsp;Col: <img style="vertical-align:middle;background:#fff" '
            f'src="data:image/png;base64,{match.last_hint_col_png_base64}">'
        )
    pregame_text = ''
    if match.pregame_riddle:
        pregame_text = f'<br>Pre-game riddle: {match.pregame_riddle}'
        if match.pregame_riddle_answer:
            pregame_text += (
                '<br><span style="background:#2ea043;color:#fff;padding:2px 8px;'
                f'border-radius:4px;font-weight:bold">LLM answer: {match.pregame_riddle_answer}</span>'
            )
    if match.free_hint_text:
        free_hint_text = f'<br>Free hint (assembled): {match.free_hint_text}'
    elif match.free_hint_fragments:
        free_hint_text = f'<br>Free hint: {len(match.free_hint_fragments)}/{match.free_hint_total} fragments received'
    else:
        free_hint_text = ''
    genesis_text = ''
    if match.genesis_team_id:
        connection_state = 'connected' if match.genesis_sim is not None else 'not connected'
        genesis_text = (
            '<br><span style="background:#0d419d;color:#fff;padding:2px 8px;border-radius:4px">'
            f'&#x1F916; Genesis: {match.genesis_team_id} ({connection_state}) &nbsp; {match.genesis_url}</span>'
        )
    status_html.value = (
        f'{flip_banner}Match #{match.match_number} &nbsp; Scores: {scores_text} &nbsp; ({pairs_text})'
        f'{turn_text}{selected_text}{memory_text}'
        f'{hint_text}{hint_images}{queued_text}{pregame_text}{free_hint_text}{genesis_text}'
    )

    cells = []
    for row in range(GRID_ROWS):
        row_cells = []
        for col in range(GRID_COLS):
            pos = pos_name(row, col)
            label = match.board_memory.get(pos, '&nbsp;')
            if pos in match.matched_positions:
                color = '#9be9a8'
            elif pos in match.board_memory:
                color = '#d9f2d9'
            else:
                color = '#f5f5f5'
            row_cells.append(f'<td style="border:1px solid #bbb;padding:6px;background:{color};min-width:70px;text-align:center">{pos}<br><small>{label}</small></td>')
        cells.append('<tr>' + ''.join(row_cells) + '</tr>')
    board_html.value = '<table style="border-collapse:collapse">' + ''.join(cells) + '</table>'

    start_button.disabled = True
    play_mode_toggle.disabled = match.game_over
    hint_button.disabled = match.game_over
    pause_button.disabled = match.game_over
    resume_button.disabled = match.game_over
    stop_button.disabled = False

    if show_raw_log_checkbox.value:
        log_lines = []
        for entry in match.log[-40:]:
            arrow = '->' if entry['direction'] == 'send' else '<-'
            log_lines.append(f"{arrow} {json.dumps(entry['message'], ensure_ascii=False)}")
        escaped_log = html.escape('\n'.join(log_lines))
        log_html.value = (
            '<pre style="margin:0;padding:8px;white-space:pre-wrap;word-break:break-word">'
            f'{escaped_log}</pre>'
        )
    else:
        log_html.value = '<i>Enable raw wire messages to view the communication log.</i>'


def on_test_connectivity_clicked(_button):
    """Quick, fast-failing check that this board can actually reach the broker
    before attempting Connect -- pynqp2p's own requests calls have no timeout,
    so a genuinely unreachable broker makes Connect hang forever instead of
    failing with a clear error. Run this first."""
    with action_output:
        clear_output(wait=True)
        server = server_text.value.strip()
        key = key_text.value.strip()
        if not server:
            print('Enter a Server address first.')
            return
        url = f'http://{server}/ping'
        print(f'Testing connectivity to {url} ...')
        start = time.time()
        try:
            response = requests.post(url, data={'key': key, 'id': 'connectivity-test'}, timeout=8)
            elapsed_ms = int((time.time() - start) * 1000)
            if response.status_code == 200:
                print(f'REACHABLE -- got {response.text!r} in {elapsed_ms}ms. Safe to Connect.')
            elif response.status_code == 401:
                print('Reached the broker, but the Key is wrong (401 Unauthorized). Check BROKER_KEY.')
            else:
                print(f'Reached the broker, but got an unexpected response: {response.status_code} {response.text!r}')
        except requests.exceptions.ConnectTimeout:
            elapsed_ms = int((time.time() - start) * 1000)
            print(
                f'NOT REACHABLE -- connection timed out after {elapsed_ms}ms. '
                'This board cannot reach the broker at all -- check network routing '
                '(relay/tunnel setup) before trying Connect, which will hang the same way.'
            )
        except requests.exceptions.ConnectionError as exc:
            print(f'NOT REACHABLE -- connection refused or DNS failure: {exc}')
        except Exception as exc:
            print(f'Connectivity test failed: {type(exc).__name__}: {exc}')


def on_connect_clicked(_button):
    global match
    with action_output:
        clear_output(wait=True)
        try:
            client = RefereeClient(
                server_text.value, key_text.value, referee_text.value, team_text.value,
                master_id=master_text.value or None,
                board_id=board_id_text.value or None,
            )
            if team_secret_text.value:
                client.join_competition(team_secret_text.value)
                print(f'Sent join_competition for team {client.team!r} (mac {client.board_id}).')
            match = MatchClient(client, on_update=render_status)
            # Start receiving messages immediately, not on the separate Start
            # click below -- join_competition() already satisfies the
            # referee's "team joined" gate at Connect time, so the pregame
            # riddle/free hint can arrive before Start is ever clicked. Every
            # autonomous action is separately gated by play_mode (off by
            # default), so it's safe to listen from Connect onward.
            match.start()
            print(f'Connected as board {client.board_id}, team {client.team!r}.')
            start_button.disabled = False
            connect_button.disabled = True
            disconnect_button.disabled = False
            render_status()
        except Exception:
            import traceback
            traceback.print_exc()


def on_disconnect_clicked(_button):
    global match
    with action_output:
        clear_output(wait=True)
        try:
            if match is not None:
                match.stop()
            match = None
            connect_button.disabled = False
            disconnect_button.disabled = True
            start_button.disabled = True
            play_mode_toggle.disabled = True
            hint_button.disabled = True
            pause_button.disabled = True
            resume_button.disabled = True
            stop_button.disabled = True
            render_status()
            print('Disconnected -- edit the connection fields above if needed, then click Connect again.')
        except Exception:
            import traceback
            traceback.print_exc()


def on_start_clicked(_button):
    with action_output:
        clear_output(wait=True)
        try:
            # Connect may happen before the operator assigns an arena. Read the
            # current field again here so the client sends to the assigned server.
            referee_id = referee_text.value.strip()
            if not referee_id:
                raise ValueError('Enter the assigned REFEREE_ID before Start (for example arena-1-referee).')
            match.client.referee_id = referee_id
            # match.start() already ran at Connect (see on_connect_clicked) --
            # this button now only arms the play controls below, it doesn't
            # start message reception.
            play_mode_toggle.disabled = False
            hint_button.disabled = False
            pause_button.disabled = False
            stop_button.disabled = False
            if control_mode_toggle.value == 'auto':
                play_mode_toggle.value = True  # triggers on_play_mode_changed -> match.set_play_mode(True)
                print('Match client started in Auto mode -- will play automatically as soon as game_start arrives.')
            else:
                print('Match client started in Wait Mode -- waiting for game_start. Switch to Play Mode when ready.')
        except Exception:
            import traceback
            traceback.print_exc()


def on_pause_clicked(_button):
    match.pause()
    pause_button.disabled = True
    resume_button.disabled = False


def on_resume_clicked(_button):
    match.resume()
    pause_button.disabled = False
    resume_button.disabled = True


def on_stop_clicked(_button):
    match.stop()
    start_button.disabled = False
    play_mode_toggle.disabled = True
    hint_button.disabled = True
    pause_button.disabled = True
    resume_button.disabled = True
    stop_button.disabled = True


def on_play_mode_changed(change):
    if match is not None:
        match.set_play_mode(change['new'])


play_mode_toggle.observe(on_play_mode_changed, names='value')


def on_hint_clicked(_button):
    obj = hint_object_text.value.strip()
    with action_output:
        clear_output(wait=True)
        if not obj:
            print('Enter an object name first.')
            return
        match.queue_hint(obj)
        print(f'Hint request sent: {obj!r}. The referee queues it automatically if this is the opponent\'s turn.')


def on_solve_riddle_clicked(_button):
    with action_output:
        clear_output(wait=True)
        if match is None or not match.pregame_riddle:
            print('No pre-game riddle received yet.')
            return
        try:
            result = solve_pregame_riddle_with_llm(match.pregame_riddle)
            match.pregame_riddle_answer = result['answer']
            print(f"LLM answer: {result['answer']}")
            render_status()
        except Exception:
            import traceback
            traceback.print_exc()


def on_camera_debug_clicked(_button):
    with camera_debug_output:
        clear_output(wait=True)
        try:
            camera_debug_one_shot(camera_debug_position_text.value)
            print(
                f'Marker orientation: TL={BORDER_MARKER_TL}, TR={BORDER_MARKER_TR}, '
                f'BR={BORDER_MARKER_BR}, BL={BORDER_MARKER_BL}; '
                f'camera rotation={BOARD_CAMERA_ROTATION_DEGREES:.1f} degrees.'
            )
        except Exception:
            import traceback
            traceback.print_exc()


solve_riddle_button.on_click(on_solve_riddle_clicked)
camera_debug_button.on_click(on_camera_debug_clicked)
test_connectivity_button.on_click(on_test_connectivity_clicked)
connect_button.on_click(on_connect_clicked)
disconnect_button.on_click(on_disconnect_clicked)
start_button.on_click(on_start_clicked)
pause_button.on_click(on_pause_clicked)
resume_button.on_click(on_resume_clicked)
stop_button.on_click(on_stop_clicked)
hint_button.on_click(on_hint_clicked)
show_raw_log_checkbox.observe(lambda _change: render_status(), names='value')

controls = widgets.VBox([
    board_clock_panel,
    widgets.HTML('<h3>Connection</h3>'),
    widgets.HBox([server_text, key_text, referee_text, master_text]),
    widgets.HBox([team_text, team_secret_text]),
    widgets.HBox([test_connectivity_button, board_id_text, connect_button, disconnect_button]),
    widgets.HTML('<h3>Camera / Grid-Cell Debug</h3>'),
    widgets.HTML(f'<small>Fixed pipeline: <b>{DETECTION_APPROACH}</b>. Corner roles are TL={BORDER_MARKER_TL}, TR={BORDER_MARKER_TR}, BR={BORDER_MARKER_BR}, BL={BORDER_MARKER_BL}. Their IDs automatically rotate and perspective-correct the board, so A1 is always top-left even when the camera is upside down. Only the oriented 416x416 selected cell is sent to YOLO. This never changes match state or contacts the referee.</small>'),
    widgets.HBox([camera_debug_position_text, camera_debug_button]),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<b>Auto-oriented board (A1 top-left)</b>'), alignment_image_widget]),
        widgets.VBox([widgets.HTML('<b>Oriented 416x416 YOLO input</b>'), processed_crop_image_widget]),
    ]),
    widgets.HTML('<b>Latest match detection in board coordinates</b>'),
    detection_image_widget,
    camera_debug_output,
    widgets.HTML('<h3>Match Control</h3>'),
    widgets.HBox([start_button, pause_button, resume_button, stop_button]),
    widgets.HTML('<small>Control: Manual uses the Wait/Play toggle below. Auto switches to Play Mode automatically as soon as Start is clicked -- no manual toggle needed.</small>'),
    control_mode_toggle,
    play_mode_toggle,
    widgets.HTML('<h3>Stage</h3>'),
    stage_html,
    widgets.HTML('<h3>Status</h3>'),
    status_html,
    board_html,
    widgets.HTML('<h3>Pre-Game Riddle</h3>'),
    widgets.HTML('<small>Auto-solved via the LLM as soon as it arrives -- this button re-solves manually if needed.</small>'),
    widgets.HBox([solve_riddle_button]),
    widgets.HTML('<h3>Hint</h3>'),
    widgets.HBox([hint_object_text, hint_button]),
    widgets.HTML('<h3>Log</h3>'),
    show_raw_log_checkbox,
    log_html,
    action_output,
    extended_tools_panel,
])

render_status()
display(controls)

### Check yourself

1. Find where the `Connect` button is created and where its click action is written.
   How are they linked together?
2. Why is there no "flip this card now" button?
